# 09 · Painel Operacional de Sincronização Manual (Google Drive)

Painel de controle manual para sincronização sob demanda entre o Kaggle Notebook e o Google Drive.

**Princípios Arquiteturais:**
- O ComfyUI gera imagens SEMPRE no SSD local (`/kaggle/working/ComfyUI/output`).
- O Google Drive é estritamente persistência/backup e não afeta o desempenho da geração.
- Cada operação (Push ou Pull por categoria) possui sua própria célula para execução individualizada.
- O sync é idempotente: arquivos idênticos (mesmo tamanho + mesmo hash SHA-256) são pulados automaticamente.

**Categorias suportadas:**
- `outputs`: Imagens e vídeos gerados
- `workflows`: Arquivos `.json` de workflows criados ou salvos
- `logs`: Logs de execução do ComfyUI
- `metadata`: Metadados e registros de geração (ignorado de forma segura quando inexistente)


In [ ]:
# 0. Configurações e Inicialização
import sys
from pathlib import Path

# Adicionar scripts do checkout atualizado ao path
REPO_DIR = Path("/kaggle/working/colab-pipeline")
SCRIPTS_DIR = Path("/kaggle/working/scripts")
if (REPO_DIR / "scripts").exists():
    sys.path.insert(0, str(REPO_DIR / "scripts"))
elif SCRIPTS_DIR.exists():
    sys.path.insert(0, str(SCRIPTS_DIR))

from kaggle_drive_sync import (
    sync_outputs, sync_category, test_drive_connection,
    get_drive_path, detect_env, DEFAULT_DRIVE_BASE
)

DRIVE_BASE = "Automa/ComfyUI"
LOCAL_OUTPUTS = Path("/kaggle/working/ComfyUI/output")
LOCAL_WORKFLOWS = Path("/kaggle/working/ComfyUI/user/default/workflows")
LOCAL_LOGS = Path("/kaggle/working/ComfyUI")
LOCAL_METADATA = Path("/kaggle/working/ComfyUI/metadata")

print(f"Ambiente: {detect_env()}")
print(f"Drive Base: {DRIVE_BASE}")
print(f"Outputs locais: {LOCAL_OUTPUTS}")
print(f"Workflows locais: {LOCAL_WORKFLOWS}")

def print_summary(stats, title=""):
    print(f"\n=== RESUMO: {title.upper()} ===")
    print(f"  ✅ Enviados/Baixados (synced): {stats.get('synced', 0)}")
    print(f"  ⏭️  Inalterados (skipped):    {stats.get('skipped', 0)}")
    print(f"  ❌ Erros (errors):          {stats.get('errors', 0)}")
    details = stats.get('details', [])
    if details:
        print("\nDetalhes dos arquivos processados:")
        for d in details[:15]:
            st = d.get('status')
            fn = d.get('file')
            sz = d.get('size_mb', 0)
            err = d.get('error', '')
            if st == 'synced':
                print(f"  [SYNC] {fn} ({sz:.2f} MB) - motivo: {d.get('reason')}")
            elif st == 'skipped':
                print(f"  [SKIP] {fn} ({sz:.2f} MB) - {d.get('reason')}")
            elif st == 'error':
                print(f"  [ERRO] {fn}: {err}")
        if len(details) > 15:
            print(f"  ... e mais {len(details) - 15} arquivo(s)")


## A) Testar Conexão e Estrutura no Google Drive

Valida rclone, permissões e estrutura remota de pastas sem transferir arquivos.

In [ ]:
# A) TESTAR DRIVE
test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
if test_res["status"] == "pass":
    print("\n✅ Conexão e estrutura do Drive validadas com sucesso!")
    drive_p = Path(test_res["drive_path"])
    for s in test_res["subdirs"]:
        print(f"  📁 {drive_p / s}")
else:
    print(f"\n❌ Falha no teste: {test_res.get('error')}")


## B) Push: Outputs locais → Google Drive

Envia imagens e vídeos gerados do SSD local para `Automa/ComfyUI/outputs/`.

In [ ]:
# B) PUSH OUTPUTS
stats = sync_category(
    category="outputs",
    action="push",
    local_outputs=LOCAL_OUTPUTS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Push Outputs")


## C) Push: Workflows locais → Google Drive

Envia arquivos de workflow `.json` para `Automa/ComfyUI/workflows/` (sem lixo ou cache).

In [ ]:
# C) PUSH WORKFLOWS
stats = sync_category(
    category="workflows",
    action="push",
    local_workflows=LOCAL_WORKFLOWS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Push Workflows")


## D) Push: Logs → Google Drive

Envia logs de execução do ComfyUI (`comfyui.log`) para `Automa/ComfyUI/logs/`.

In [ ]:
# D) PUSH LOGS
stats = sync_category(
    category="logs",
    action="push",
    local_logs=LOCAL_LOGS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Push Logs")


## E) Push: Metadata → Google Drive

Envia metadados estruturados (se existirem) para `Automa/ComfyUI/metadata/`.

In [ ]:
# E) PUSH METADATA
stats = sync_category(
    category="metadata",
    action="push",
    local_metadata=LOCAL_METADATA,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Push Metadata")


## F) Pull: Workflows do Google Drive → Local

Baixa workflows salvos no Drive para o diretório local do ComfyUI.

In [ ]:
# F) PULL WORKFLOWS
stats = sync_category(
    category="workflows",
    action="pull",
    local_workflows=LOCAL_WORKFLOWS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Pull Workflows")


## G) Pull: Outputs do Google Drive → Local (Opcional)

Restaura imagens salvas no Drive para a pasta local de outputs sob demanda.

In [ ]:
# G) PULL OUTPUTS (Opcional)
stats = sync_category(
    category="outputs",
    action="pull",
    local_outputs=LOCAL_OUTPUTS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Pull Outputs")


## H) Pull: Logs do Google Drive → Local (Opcional)

Baixa histórico de logs do Drive para inspeção local.

In [ ]:
# H) PULL LOGS (Opcional)
stats = sync_category(
    category="logs",
    action="pull",
    local_logs=LOCAL_LOGS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Pull Logs")


## I) Pull: Metadata do Google Drive → Local (Opcional)

Baixa arquivos de metadata salvos no Drive.

In [ ]:
# I) PULL METADATA (Opcional)
stats = sync_category(
    category="metadata",
    action="pull",
    local_metadata=LOCAL_METADATA,
    drive_base=DRIVE_BASE,
    env="kaggle",
)
print_summary(stats, "Pull Metadata")


## Visualizar Conteúdo Atual no Google Drive

Exibe a listagem dos arquivos e tamanho ocupado por categoria no Google Drive.

In [ ]:
# Listar arquivos remotos no Google Drive
try:
    drive_path = get_drive_path(drive_base=DRIVE_BASE, env="kaggle")
    print(f"Drive path: {drive_path}")
    for subdir in ["outputs", "workflows", "logs", "metadata"]:
        d = drive_path / subdir
        if d.exists():
            flist = [f for f in d.rglob("*") if f.is_file()]
            total_mb = sum(f.stat().st_size for f in flist) / (1024 ** 2)
            print(f"\n📁 {subdir}: {len(flist)} arquivo(s), {total_mb:.2f} MB total")
            for f in sorted(flist)[:5]:
                sz_mb = f.stat().st_size / (1024 ** 2)
                print(f"   - {f.name} ({sz_mb:.2f} MB)")
            if len(flist) > 5:
                print(f"   ... e mais {len(flist) - 5} arquivo(s)")
        else:
            print(f"\n📁 {subdir}: (vazio / não criado)")
except Exception as e:
    print(f"[ERROR] Não foi possível inspecionar o Drive: {e}")
